# Week 5: Hospital General Information (Bronze, Silver, Gold join)

Purpose: add the second CMS dataset from the proposal (LO1), so H2 (quality), H3 (ownership) and the Week 6 regression can run.

Source: CMS Provider Data Catalog, *Hospital General Information* (dataset xubh-q36u), file `Hospital_General_Information.csv`, downloaded 2026-09-04. 5,419 hospitals x 38 columns. Join key: `Facility ID`, the 6-character CCN.

Outputs: `workspace.cms.bronze_hospital_info`, `workspace.cms.silver_hospital_info`, `workspace.cms.gold_dim_hospital_profile` (one row per inpatient hospital, joined to Gold on `ccn`).

Decisions (2026-11-02, logged in docs/data_quality_log.md)

1. Snapshot timing: use the current file. Ratings and ownership are a 2026 snapshot against CY2024 claims; logged as a limitation (DQ-24).

2. "Not Available" star ratings: kept as their own "Unrated" category. A numeric rating column (NULL for Unrated) is stored next to it so rated hospitals can still be ranked.

3. Ownership: 3 groups. For-profit = Proprietary + Physician. Non-profit = Voluntary non-profit (Private, Other, Church). Government = Local, Hospital District or Authority, State, Federal, Tribal. The original CMS value is kept in Silver.

4. Unmatched inpatient hospitals (39 expected): kept with NULL attributes and an `is_profile_matched` flag; excluded only from ownership and quality tests (DQ-25).

5. Gold design: a separate one-row-per-hospital table, `gold_dim_hospital_profile`, built here. `gold_dim_hospital` (notebook 03) is not modified, so every notebook stays rerunnable on its own.

In [ ]:
# ---------------------------------------------------------------
# Cell 2: Config
# Purpose: every path, table name and expected value in one place
# Inputs: none | Outputs: Python variables used by later cells
# Expected values come from local profiling of the same file (2026-11-02)
# ---------------------------------------------------------------
CATALOG = "workspace"
SCHEMA = "cms"
VOLUME = "raw"
FILE_NAME = "Hospital_General_Information.csv"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_hospital_info"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_hospital_info"

EXPECTED_ROWS = 5_419
EXPECTED_SOURCE_COLS = 38

print("Source:", SOURCE_PATH)
print("Target:", BRONZE_TABLE)

In [ ]:
# ---------------------------------------------------------------
# Cell 3: Confirm the file landed in the volume
# Purpose: fail early if the CSV was not uploaded
# Inputs: /Volumes/workspace/cms/raw/ | Outputs: file listing
# Expected: 2 files (Hospital_General_Information.csv and the inpatient CSV)
# ---------------------------------------------------------------
files = dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/")
display(files)
assert any(f.name == FILE_NAME for f in files), f"{FILE_NAME} not found in the volume"

In [ ]:
# ---------------------------------------------------------------
# Cell 4: Bronze ingest
# Purpose: land the CMS file as-is, every column STRING, plus lineage
# Inputs: SOURCE_PATH | Outputs: workspace.cms.bronze_hospital_info
# Expected: 5,419 rows, 38 source columns, 0 corrupt records
# Same pattern as 01_bronze_ingest.py
# ---------------------------------------------------------------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# Step A: read only the header row to get the column names exactly as CMS wrote them
header_cols = spark.read.option("header", True).csv(SOURCE_PATH).columns
assert len(header_cols) == EXPECTED_SOURCE_COLS, f"Expected {EXPECTED_SOURCE_COLS} columns, got {len(header_cols)}"

# Step B: explicit all-string schema, so Facility ID keeps its leading zeros
# and any malformed line lands in _corrupt_record instead of being dropped
bronze_schema = StructType(
    [StructField(c, StringType(), True) for c in header_cols]
    + [StructField("_corrupt_record", StringType(), True)]
)

raw_df = (
    spark.read
    .schema(bronze_schema)
    .option("header", True)
    .option("enforceSchema", False)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", False)
    .option("encoding", "UTF-8")
    .csv(SOURCE_PATH)
)

# Step C: add lineage columns, the only things Bronze adds
bronze_df = (
    raw_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

spark.sql(f"""
COMMENT ON TABLE {BRONZE_TABLE} IS
'Bronze: CMS Hospital General Information (Provider Data Catalog xubh-q36u, downloaded 2026-09-04), loaded as-is. All source columns are STRING. One row = one hospital (Facility ID = CCN). Lineage: _ingest_ts, _source_file.'
""")

# Step D: row count and corrupt records
bronze = spark.table(BRONZE_TABLE)
n_rows = bronze.count()
n_corrupt = bronze.filter(F.col("_corrupt_record").isNotNull()).count()
print("Rows:", n_rows, "| Corrupt records:", n_corrupt)
assert n_rows == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, got {n_rows}"
assert n_corrupt == 0, f"{n_corrupt} corrupt records"
print("Wrote", BRONZE_TABLE)

In [ ]:
%sql
--- ---------------------------------------------------------------
--- Cell 5: Bronze checks
--- Purpose: confirm the key column is complete, unique and 6 characters
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: one result row
--- Expected: 5,419 rows; 5,419 distinct Facility IDs; 0 NULL IDs; 0 IDs not 6 characters
--- ---------------------------------------------------------------
--- TODO: one SELECT with COUNT(*), COUNT(DISTINCT `Facility ID`), and two SUM(CASE ...) checks
---       (column names with spaces need backticks)

In [ ]:
%sql
--- ---------------------------------------------------------------
--- Cell 6: Category profile
--- Purpose: see every value of the columns the analysis uses (decisions 2 and 3)
--- Inputs: workspace.cms.bronze_hospital_info | Outputs: value counts
--- Expected: 8 Hospital Type values, 12 Hospital Ownership values,
---           ratings 1 to 5 plus "Not Available" (2,245)
--- ---------------------------------------------------------------
--- TODO: GROUP BY counts for `Hospital Type`, `Hospital Ownership` and `Hospital overall rating`

In [ ]:
%sql
--- ---------------------------------------------------------------
--- Cell 7: Join preview
--- Purpose: how many inpatient hospitals find a match before Silver is built
--- Inputs: workspace.cms.gold_dim_hospital, workspace.cms.bronze_hospital_info | Outputs: counts
--- Expected: 2,906 inpatient hospitals; 2,867 matched; 39 unmatched
--- Silver ccn is already 6 digits (DQ-02), so no LPAD is needed
--- ---------------------------------------------------------------
--- TODO: LEFT JOIN gold_dim_hospital to Bronze on ccn = `Facility ID`; count matched and unmatched

## Silver

Next: type the rating (numeric column, NULL for Unrated, plus a rating category with "Unrated"), map ownership to the three groups, keep the original CMS columns for traceability, and add checks.